# Sentence length analysis: do any test sentences exceed 512 tokens?

BERT-family models (SecureBERT et al.) cap input at 512 tokens. Here we check, for the
cleaned test split of each of the four datasets, how often a single sentence exceeds that
limit — both by raw word count and by SecureBERT **subword** count (the count that actually
matters for the model).

Sentences are already split (one sentence per blank-line-separated block in the
`.cleaned`/`.unified` files), so this is a per-sentence measurement, not per-document.

In [ ]:
import pandas as pd
from transformers import AutoTokenizer

from nlp_cyber_ner.dataset import read_iob2_file

tokenizer = AutoTokenizer.from_pretrained("Cyber-ThreaD/SecureBERT-DNRTI")

MAX_LEN = 512

# cyner has no `.cleaned` (it goes raw -> processed), so we use its unified test split;
# the other three use their cleaned test files.
test_files = {
    "cyner": "../data/processed/cyner/test.unified",
    "aptner": "../data/interim/APTNer/APTNERtest.cleaned",
    "attacker": "../data/interim/attacker/test.cleaned",
    "dnrti": "../data/interim/DNRTI/test.cleaned",
}

rows = []
for name, path in test_files.items():
    data = read_iob2_file(path)
    word_lens = [len(words) for words, _ in data]
    subword_lens = [
        len(tokenizer(words, is_split_into_words=True)["input_ids"]) for words, _ in data
    ]
    rows.append(
        {
            "dataset": name,
            "sentences": len(data),
            "words>512": sum(n > MAX_LEN for n in word_lens),
            "subwords>512": sum(n > MAX_LEN for n in subword_lens),
            "max_words": max(word_lens),
            "max_subwords": max(subword_lens),
        }
    )

summary = pd.DataFrame(rows).set_index("dataset")
summary

## Conclusion

**No test sentence in any of the four datasets exceeds 512 tokens** — neither by word count
nor by SecureBERT subword count. The longest single sentence anywhere is in APTNER at
113 words / 226 subwords, comfortably under half the limit.

Consequently the `truncation=True, max_length=512` in the prediction loop never truncates on
this test data, and the `"O"`-padding fallback for dropped trailing words is dead code in
practice.